# Fabric Workspace Inventory v3.1 – Production Notebook

### Data sources

| API | Base URL | Purpose | Permission |
|-----|----------|---------|------------|
| Core Items | `api.fabric.microsoft.com/v1` | Item list (name, type, ID) | Viewer |
| Admin Items | `api.fabric.microsoft.com/v1` | Owner, lastUpdatedDate, state | Fabric Admin |
| Scanner API | `sempy_labs.admin.scan_workspaces()` | Created date, modified by | Fabric Admin |
| Activity Events | `api.powerbi.com/v1.0/myorg` | Genuine last-used / access data | Fabric Admin |
| Unused Artifacts | `sempy_labs.admin.list_unused_artifacts()` | Unused detection + created/last-accessed dates | Fabric Admin |

> **Important:** `Last Modified ≠ Last Used`. This notebook keeps these strictly separate.


## 1. Install dependencies

In [ ]:
# ── Install semantic-link-labs if not already available ─────
# This cell must run BEFORE all other code cells.
# %pip may trigger a kernel restart — if so, re-run from the next cell onward.
try:
    import sempy_labs
    print("semantic-link-labs is available.")
except ImportError:
    print("Installing semantic-link-labs...")
    %pip install semantic-link-labs -q
    print("Installed. If the kernel restarted, re-run all cells from here.")


## 2. Parameters

In [ ]:
# ═══════════════════════════════════════════════════════════════
# PARAMETERS — edit these before running
# ═══════════════════════════════════════════════════════════════

workspace_id             = ""          # Leave blank to use the current workspace
save_to_lakehouse        = True        # Append snapshot to a Delta table
lakehouse_table_name     = "workspace_inventory_snapshot"
stale_cutoff_days        = 90          # Flag items not modified AND not used in this many days
activity_lookback_days   = 30          # How many days of activity events to scan (max 30)

# ── Optional enrichment (require semantic-link-labs) ───────
enable_scanner_api       = True        # Get created_date & modified_by via Scanner API
enable_unused_artifacts  = True        # Detect unused Power BI items + extract dates

# ── Activity Events ───────────────────────────────────────
enable_activity_events   = True        # Genuine usage data from Power BI Activity Events API


## 3. Setup and authentication

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import uuid
import logging
from datetime import datetime, timezone, timedelta
import notebookutils

# ── Structured logging ─────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("fabric_inventory")

# ── Constants ──────────────────────────────────────────────
FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"
POWERBI_API_BASE = "https://api.powerbi.com/v1.0/myorg"
FABRIC_APP = "https://app.fabric.microsoft.com"

# ── Snapshot ID (idempotent) ───────────────────────────────
snapshot_id       = str(uuid.uuid4())
snapshot_time_utc = datetime.now(timezone.utc).isoformat()
log.info(f"Snapshot ID: {snapshot_id}")
log.info(f"Snapshot time: {snapshot_time_utc}")

# ── Timing tracker ─────────────────────────────────────────
section_times = {}
def timed_section(name):
    class _Timer:
        def __enter__(self):
            self.start = time.time()
            log.info(f"▶ Starting: {name}")
            return self
        def __exit__(self, *args):
            elapsed = time.time() - self.start
            section_times[name] = elapsed
            log.info(f"✔ Completed: {name} ({elapsed:.1f}s)")
    return _Timer()


In [ ]:
with timed_section("Authentication"):
    if not workspace_id:
        workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    log.info(f"Target workspace: {workspace_id}")

    token = notebookutils.credentials.getToken("pbi")
    token_acquired_at = time.time()

    def get_headers():
        global token, token_acquired_at
        if time.time() - token_acquired_at > 2400:
            log.info("Refreshing authentication token...")
            token = notebookutils.credentials.getToken("pbi")
            token_acquired_at = time.time()
        return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    log.info("Authentication successful.")


## 4. API helpers

In [ ]:
def call_fabric_api(url, max_retries=5, caller="API"):
    """GET wrapper for Fabric REST API (api.fabric.microsoft.com)."""
    for attempt in range(max_retries):
        resp = requests.get(url, headers=get_headers())
        if resp.status_code == 200:
            return resp.json()
        elif resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            log.warning(f"[{caller}] Throttled. Waiting {wait}s (attempt {attempt+1})...")
            time.sleep(wait)
        elif resp.status_code in (401, 403):
            raise PermissionError(f"[{caller}] Access denied ({resp.status_code}): {resp.text[:300]}")
        else:
            log.error(f"[{caller}] HTTP {resp.status_code}: {resp.text[:300]}")
            resp.raise_for_status()
    raise RuntimeError(f"[{caller}] Failed after {max_retries} retries: {url}")


def call_powerbi_api(url, max_retries=5, caller="API"):
    """GET wrapper for Power BI REST API (api.powerbi.com). Same token works."""
    for attempt in range(max_retries):
        resp = requests.get(url, headers=get_headers())
        if resp.status_code == 200:
            return resp.json()
        elif resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            log.warning(f"[{caller}] Throttled. Waiting {wait}s (attempt {attempt+1})...")
            time.sleep(wait)
        elif resp.status_code in (401, 403):
            raise PermissionError(f"[{caller}] Access denied ({resp.status_code}): {resp.text[:300]}")
        else:
            log.error(f"[{caller}] HTTP {resp.status_code}: {resp.text[:300]}")
            resp.raise_for_status()
    raise RuntimeError(f"[{caller}] Failed after {max_retries} retries: {url}")


## 5. Workspace metadata

In [ ]:
with timed_section("Workspace metadata"):
    workspace_name = ""
    try:
        ws_info = call_fabric_api(
            f"{FABRIC_API_BASE}/workspaces/{workspace_id}", caller="Workspace"
        )
        workspace_name = ws_info.get("displayName", "")
        log.info(f"Workspace name: {workspace_name}")
    except Exception as e:
        log.warning(f"Could not retrieve workspace name: {e}")


## 6. Layer 1 — Core Items API

```
GET /v1/workspaces/{workspaceId}/items
```


In [ ]:
with timed_section("Core Items API"):
    def get_core_items(ws_id):
        items, url = [], f"{FABRIC_API_BASE}/workspaces/{ws_id}/items"
        while url:
            data = call_fabric_api(url, caller="CoreItems")
            items.extend(data.get("value", []))
            cont = data.get("continuationToken")
            url = (f"{FABRIC_API_BASE}/workspaces/{ws_id}/items"
                   f"?continuationToken={cont}") if cont else None
        return items

    core_items = get_core_items(workspace_id)
    log.info(f"Core Items API returned {len(core_items)} items.")

    if core_items:
        df_core = pd.DataFrame(core_items)
        df_core = df_core[["id", "displayName", "type", "description"]].rename(
            columns={"displayName": "name"}
        )
    else:
        df_core = pd.DataFrame(columns=["id", "name", "type", "description"])
        log.warning("Core Items API returned 0 items.")


## 7. Layer 2 — Admin Items API

```
GET /v1/admin/items?workspaceId={workspaceId}
```

> The Admin Items API does **NOT** return `createdDate` or `modifiedBy`. Those come from the Scanner API.


In [ ]:
admin_available = False
df_admin = pd.DataFrame()
admin_only_ids = set()
admin_items_raw = []

with timed_section("Admin Items API"):
    try:
        def extract_principal(principal_obj):
            if not principal_obj or not isinstance(principal_obj, dict):
                return None
            user = principal_obj.get("userDetails", {})
            return (user.get("userPrincipalName")
                    or principal_obj.get("displayName")
                    or principal_obj.get("id"))

        def get_admin_items(ws_id):
            items, url = [], f"{FABRIC_API_BASE}/admin/items?workspaceId={ws_id}"
            while url:
                data = call_fabric_api(url, caller="AdminItems")
                items.extend(data.get("itemEntities", []))
                cont = data.get("continuationToken")
                url = (f"{FABRIC_API_BASE}/admin/items?workspaceId={ws_id}"
                       f"&continuationToken={cont}") if cont else None
            return items

        admin_items_raw = get_admin_items(workspace_id)
        log.info(f"Admin Items API returned {len(admin_items_raw)} items.")
        admin_available = True

        if admin_items_raw:
            log.info(f"Admin API sample keys: {list(admin_items_raw[0].keys())}")

        df_admin = pd.DataFrame([{
            "id":            i.get("id"),
            "created_by":    extract_principal(i.get("creatorPrincipal")),
            "last_modified": i.get("lastUpdatedDate"),
            "state":         i.get("state"),
            "capacity_id":   i.get("capacityId"),
        } for i in admin_items_raw])

        admin_only_ids = set(df_admin["id"].dropna()) - set(df_core["id"].dropna())
        if admin_only_ids:
            log.info(f"{len(admin_only_ids)} items in Admin API but not in Core API (system-generated).")

    except PermissionError as e:
        log.warning("Admin API not accessible — not a Fabric Administrator.")
        log.warning(f"  Detail: {e}")
        log.info("Continuing with Core Items data only.")


## 8. Scanner API — created_date and modified_by

The Admin Items API does **not** return `createdDate` or `modifiedBy`. The Scanner API (via `semantic-link-labs`) provides these.

The function signature is:
```python
scan_workspaces(workspace=workspace_id, ...)
```


In [ ]:
scanner_created = {}   # item_id -> created_date
scanner_modified = {}  # item_id -> modified_by

with timed_section("Scanner API (created_date, modified_by)"):
    if not enable_scanner_api:
        log.info("Scanner API disabled via parameter.")
    elif not admin_available:
        log.info("Skipping — requires Fabric Administrator.")
    else:
        try:
            import sempy_labs.admin as sll_admin

            log.info(f"Calling scan_workspaces(workspace='{workspace_id}')...")

            # ── Try keyword argument variations ────────────
            # The function signature varies across sempy_labs versions.
            df_scan = None
            scan_errors = []

            # Attempt 1: workspace= keyword
            try:
                df_scan = sll_admin.scan_workspaces(workspace=workspace_id)
            except TypeError as e1:
                scan_errors.append(f"workspace= : {e1}")

            # Attempt 2: No argument (scans all workspaces user has access to)
            if df_scan is None:
                try:
                    df_scan = sll_admin.scan_workspaces()
                except Exception as e2:
                    scan_errors.append(f"no args: {e2}")

            if df_scan is None or (hasattr(df_scan, 'empty') and df_scan.empty):
                log.warning(f"scan_workspaces() returned no data. Attempts: {scan_errors}")
            else:
                log.info(f"Scanner API returned {len(df_scan)} items.")
                log.info(f"Scanner API columns: {list(df_scan.columns)}")

                # ── Filter to our workspace if we scanned all ──
                ws_col = None
                for candidate in ["Workspace Id", "workspaceId", "workspace_id"]:
                    if candidate in df_scan.columns:
                        ws_col = candidate
                        break
                if ws_col:
                    before = len(df_scan)
                    df_scan = df_scan[df_scan[ws_col].astype(str) == workspace_id]
                    log.info(f"  Filtered to workspace: {len(df_scan)} items (from {before})")

                # ── Find columns ───────────────────────────
                id_col = None
                for c in ["Id", "id", "ID"]:
                    if c in df_scan.columns:
                        id_col = c
                        break

                created_col = None
                for c in ["Created Date", "createdDateTime", "created_date", "CreatedDate", "createdDate", "Created Date Time"]:
                    if c in df_scan.columns:
                        created_col = c
                        break

                modified_by_col = None
                for c in ["Modified By", "modifiedBy", "modified_by", "ModifiedBy", "lastModifiedBy", "configuredBy", "Configured By"]:
                    if c in df_scan.columns:
                        modified_by_col = c
                        break

                if id_col:
                    for _, row in df_scan.iterrows():
                        item_id = str(row[id_col]) if pd.notna(row.get(id_col)) else None
                        if not item_id:
                            continue
                        if created_col and pd.notna(row.get(created_col)):
                            scanner_created[item_id] = str(row[created_col])
                        if modified_by_col and pd.notna(row.get(modified_by_col)):
                            scanner_modified[item_id] = str(row[modified_by_col])

                    log.info(f"  created_date populated for {len(scanner_created)} items.")
                    log.info(f"  modified_by populated for {len(scanner_modified)} items.")
                else:
                    log.warning(f"Could not find ID column. Available: {list(df_scan.columns)}")

        except ImportError:
            log.info("semantic-link-labs not installed. created_date and modified_by will be null.")
        except Exception as e:
            log.warning(f"Scanner API failed: {e}")
            log.info("Continuing without created_date and modified_by.")


## 9. Merge and enrich

In [ ]:
with timed_section("Merge and enrich"):
    # ── Outer join Core + Admin ────────────────────────────
    if admin_available and not df_admin.empty:
        admin_name_map = {}
        for ai in admin_items_raw:
            if ai.get("id") in admin_only_ids:
                admin_name_map[ai["id"]] = {
                    "name": ai.get("name", ai.get("displayName", "")),
                    "type": ai.get("type", "Unknown"),
                    "description": ai.get("description", ""),
                }

        df_final = df_core.merge(df_admin, on="id", how="outer")

        for idx, row in df_final[df_final["name"].isna()].iterrows():
            info = admin_name_map.get(row["id"], {})
            for col, val in info.items():
                if val:
                    df_final.at[idx, col] = val
    else:
        df_final = df_core.copy()
        for col in ["created_by", "last_modified", "state", "capacity_id"]:
            if col not in df_final.columns:
                df_final[col] = None

    # ── Add Scanner API fields ─────────────────────────────
    df_final["created_date"] = df_final["id"].map(
        lambda x: scanner_created.get(str(x)) if pd.notna(x) else None
    )
    df_final["modified_by"] = df_final["id"].map(
        lambda x: scanner_modified.get(str(x)) if pd.notna(x) else None
    )

    # ── Workspace context ──────────────────────────────────
    df_final["workspace_id"]   = workspace_id
    df_final["workspace_name"] = workspace_name

    # ── Construct web URL ──────────────────────────────────
    type_url_map = {
        "Report": "reports", "SemanticModel": "datasets", "Dashboard": "dashboards",
        "Dataflow": "dataflows", "DataPipeline": "pipelines", "Notebook": "notebooks",
        "Lakehouse": "lakehouses", "Warehouse": "warehouses", "SQLEndpoint": "sqlEndpoints",
        "Eventhouse": "eventhouses", "KQLDatabase": "kqlDatabases", "KQLQueryset": "kqlQuerysets",
        "KQLDashboard": "kqlDashboards", "Environment": "environments", "SQLDatabase": "sqlDatabases",
        "MirroredDatabase": "mirroredDatabases", "Eventstream": "eventstreams", "Reflex": "reflexes",
        "CopyJob": "copyJobs", "SparkJobDefinition": "sparkJobDefinitions",
    }
    def build_web_url(row):
        seg = type_url_map.get(row.get("type"))
        if seg and row.get("id"):
            return f"{FABRIC_APP}/groups/{workspace_id}/{seg}/{row['id']}"
        return None
    df_final["web_url"] = df_final.apply(build_web_url, axis=1)

    # ── Snapshot metadata ──────────────────────────────────
    df_final["snapshot_id"]       = snapshot_id
    df_final["snapshot_time_utc"] = snapshot_time_utc

    df_final = df_final.sort_values(["type", "name"]).reset_index(drop=True)

    log.info(f"Total items in inventory: {len(df_final)}")
    if admin_available:
        log.info(f"  Core API items:   {len(df_core)}")
        log.info(f"  Admin-only items: {len(admin_only_ids)}")
    log.info(f"  created_date populated: {df_final['created_date'].notna().sum()}")
    log.info(f"  modified_by populated:  {df_final['modified_by'].notna().sum()}")


## 10. Type summary

In [ ]:
with timed_section("Type summary"):
    summary = (
        df_final.groupby("type").size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    print(f"\nObject counts by type ({len(df_final)} total items, {summary['type'].nunique()} types):\n")
    display(summary)


## 11. Unused artifacts + date enrichment

`list_unused_artifacts()` returns:
- `Artifact Id` — used to flag unused items
- `Created Date Time` — **epoch ms** → converted to ISO datetime → fills `created_date` for these items
- `Last Accessed Date Time` — **epoch ms** → converted to ISO datetime → fills `last_used_date` for these items

This partially fills the `created_date` gap for Power BI items even when the Scanner API fails.


In [ ]:
unused_artifact_ids = set()
unused_created_dates = {}    # artifact_id -> created_date ISO string
unused_last_accessed = {}    # artifact_id -> last_accessed ISO string

with timed_section("Unused artifacts (semantic-link-labs)"):
    if not enable_unused_artifacts:
        log.info("Unused artifact detection disabled via parameter.")
    elif not admin_available:
        log.info("Skipping — requires Fabric Administrator.")
    else:
        try:
            import sempy_labs.admin as sll_admin

            log.info("Calling list_unused_artifacts()...")
            df_unused = sll_admin.list_unused_artifacts()

            if df_unused is None or (hasattr(df_unused, 'empty') and df_unused.empty):
                log.info("list_unused_artifacts() returned 0 items.")
            else:
                log.info(f"list_unused_artifacts() returned {len(df_unused)} items.")
                log.info(f"  Columns: {list(df_unused.columns)}")

                # ── Find ID column ─────────────────────────
                id_col = None
                for c in ["Artifact Id", "Id", "id", "artifactId"]:
                    if c in df_unused.columns:
                        id_col = c
                        break

                if id_col:
                    unused_artifact_ids = set(df_unused[id_col].dropna().astype(str))
                    log.info(f"  Matched {len(unused_artifact_ids)} item IDs as unused.")
                else:
                    log.warning(f"  Could not find ID column. Available: {list(df_unused.columns)}")

                # ── Extract and convert epoch timestamps ───
                def epoch_ms_to_iso(val):
                    """Convert epoch milliseconds (int/float) to ISO datetime string."""
                    try:
                        if pd.notna(val) and val != "" and val != "None":
                            ts = pd.to_datetime(float(val), unit="ms", utc=True)
                            return ts.isoformat()
                    except (ValueError, TypeError, OverflowError):
                        pass
                    return None

                created_col = None
                for c in ["Created Date Time", "createdDateTime", "Created Date"]:
                    if c in df_unused.columns:
                        created_col = c
                        break

                accessed_col = None
                for c in ["Last Accessed Date Time", "lastAccessedDateTime", "Last Accessed"]:
                    if c in df_unused.columns:
                        accessed_col = c
                        break

                if id_col:
                    for _, row in df_unused.iterrows():
                        aid = str(row[id_col]) if pd.notna(row.get(id_col)) else None
                        if not aid:
                            continue
                        if created_col:
                            dt = epoch_ms_to_iso(row.get(created_col))
                            if dt:
                                unused_created_dates[aid] = dt
                        if accessed_col:
                            dt = epoch_ms_to_iso(row.get(accessed_col))
                            if dt:
                                unused_last_accessed[aid] = dt

                    log.info(f"  Extracted created_date for {len(unused_created_dates)} items.")
                    log.info(f"  Extracted last_accessed for {len(unused_last_accessed)} items.")

                if len(df_unused) > 0:
                    display(df_unused)

        except ImportError:
            log.info("semantic-link-labs not installed. Skipping unused artifact detection.")
        except Exception as e:
            log.warning(f"list_unused_artifacts() failed: {e}")
            log.info("Continuing without unused artifact data.")


## 12. Activity Events API

```
GET https://api.powerbi.com/v1.0/myorg/admin/activityevents
    ?startDateTime='YYYY-MM-DDT00:00:00.000Z'
    &endDateTime='YYYY-MM-DDT23:59:59.000Z'
```

Start and end must fall within the **same UTC day**. Max 30 days, max 200 requests/hour.


In [ ]:
activity_data = {}

with timed_section("Activity Events API"):
    if not enable_activity_events:
        log.info("Activity Events disabled via parameter.")
    elif not admin_available:
        log.info("Skipping — requires Fabric Administrator.")
    else:
        try:
            # Use yesterday as end to avoid partial-day edge issues
            end_dt   = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
            start_dt = end_dt - timedelta(days=min(activity_lookback_days, 30))
            log.info(f"Scanning activity events: {start_dt.date()} → {end_dt.date()}")
            log.info(f"Endpoint: {POWERBI_API_BASE}/admin/activityevents")

            all_events = []
            current_day = start_dt
            days_scanned = 0
            days_failed = 0
            first_event_logged = False

            while current_day < end_dt:
                day_str = current_day.strftime("%Y-%m-%d")
                s_str = f"{day_str}T00:00:00.000Z"
                e_str = f"{day_str}T23:59:59.000Z"

                url = (f"{POWERBI_API_BASE}/admin/activityevents"
                       f"?startDateTime='{s_str}'&endDateTime='{e_str}'")

                day_ok = True
                while url:
                    try:
                        data = call_powerbi_api(url, caller="ActivityEvents")
                    except PermissionError:
                        log.warning("Activity Events API not accessible.")
                        raise
                    except Exception as e:
                        # Silently skip days that fail (edge of retention window)
                        days_failed += 1
                        day_ok = False
                        break

                    events = data.get("activityEventEntities", [])
                    all_events.extend(events)

                    if events and not first_event_logged:
                        log.info(f"  Sample event keys: {list(events[0].keys())[:15]}")
                        first_event_logged = True

                    cont_uri = data.get("continuationUri")
                    url = cont_uri if cont_uri else None

                current_day += timedelta(days=1)
                days_scanned += 1

                if days_scanned % 10 == 0:
                    log.info(f"  Scanned {days_scanned} days, {len(all_events)} events so far...")

            log.info(f"Retrieved {len(all_events)} events over {days_scanned} days ({days_failed} days failed/skipped).")

            # ── Aggregate per item ─────────────────────────
            if all_events:
                df_events = pd.DataFrame(all_events)

                id_col = None
                for c in ["ArtifactId", "artifactId", "ItemId", "itemId"]:
                    if c in df_events.columns:
                        id_col = c
                        break

                time_col = None
                for c in ["CreationTime", "creationTime", "Timestamp", "timestamp"]:
                    if c in df_events.columns:
                        time_col = c
                        break

                user_col = None
                for c in ["UserId", "userId", "UserKey", "userKey"]:
                    if c in df_events.columns:
                        user_col = c
                        break

                if id_col and time_col:
                    df_events["_item_id"] = df_events[id_col].astype(str)
                    df_events["_time"]    = pd.to_datetime(df_events[time_col], errors="coerce", utc=True)

                    for item_id, grp in df_events.groupby("_item_id"):
                        last_used = grp["_time"].max()
                        activity_data[item_id] = {
                            "last_used":    last_used.isoformat() if pd.notna(last_used) else None,
                            "access_count": len(grp),
                            "unique_users": grp[user_col].nunique() if user_col else None,
                        }

                    log.info(f"Aggregated usage data for {len(activity_data)} distinct items.")
                else:
                    log.warning(f"Could not identify ID/Time columns. Found: {list(df_events.columns)}")

        except PermissionError:
            log.warning("Activity Events API requires Fabric Administrator. Skipping.")
        except Exception as e:
            log.warning(f"Activity Events processing failed: {e}")
            log.info("Continuing without activity data.")


## 13. Merge usage data into inventory

In [ ]:
with timed_section("Merge usage data"):
    # ── Activity Events columns ────────────────────────────
    df_final["last_used_date"] = df_final["id"].map(
        lambda x: activity_data.get(str(x), {}).get("last_used") if pd.notna(x) else None
    )
    df_final["access_count_30d"] = df_final["id"].map(
        lambda x: activity_data.get(str(x), {}).get("access_count") if pd.notna(x) else None
    )
    df_final["unique_users_30d"] = df_final["id"].map(
        lambda x: activity_data.get(str(x), {}).get("unique_users") if pd.notna(x) else None
    )

    # ── Unused artifact flag ───────────────────────────────
    if unused_artifact_ids:
        df_final["is_unused_artifact"] = df_final["id"].apply(
            lambda x: str(x) in unused_artifact_ids if pd.notna(x) else False
        )
    else:
        df_final["is_unused_artifact"] = False

    # ── Enrich dates from unused artifacts ─────────────────
    # Fill created_date from unused artifacts where Scanner API didn't provide it
    if unused_created_dates:
        for idx, row in df_final.iterrows():
            item_id = str(row["id"]) if pd.notna(row.get("id")) else None
            if item_id and pd.isna(row.get("created_date")):
                dt = unused_created_dates.get(item_id)
                if dt:
                    df_final.at[idx, "created_date"] = dt

        log.info(f"created_date after unused-artifacts enrichment: {df_final['created_date'].notna().sum()}")

    # Fill last_used_date from unused artifacts where Activity Events didn't provide it
    # Note: "Last Accessed" from unused artifacts is the LAST access before the item became unused
    if unused_last_accessed:
        for idx, row in df_final.iterrows():
            item_id = str(row["id"]) if pd.notna(row.get("id")) else None
            if item_id and pd.isna(row.get("last_used_date")):
                dt = unused_last_accessed.get(item_id)
                if dt:
                    df_final.at[idx, "last_used_date"] = dt

        log.info(f"last_used_date after unused-artifacts enrichment: {df_final['last_used_date'].notna().sum()}")

    # ── Days since last used ───────────────────────────────
    now_utc = pd.Timestamp.now(tz="UTC")

    last_used_dt = pd.to_datetime(df_final["last_used_date"], errors="coerce", utc=True)
    df_final["days_since_last_used"] = (
        (now_utc - last_used_dt).dt.days.where(last_used_dt.notna(), other=None)
    )

    # ── Days since modified ────────────────────────────────
    last_mod_dt = pd.to_datetime(df_final["last_modified"], errors="coerce", utc=True)
    df_final["days_since_modified"] = (
        (now_utc - last_mod_dt).dt.days.where(last_mod_dt.notna(), other=None)
    )

    # ── Summary ────────────────────────────────────────────
    log.info(f"Items with usage data (Activity Events): {(df_final['access_count_30d'].notna()).sum()}")
    log.info(f"Items with last_used_date (any source):  {df_final['last_used_date'].notna().sum()}")
    log.info(f"Items flagged as unused artifacts:        {df_final['is_unused_artifact'].sum()}")


## 14. Governance analysis

`is_stale` is computed using `np.select` to guarantee every row gets True or False — no NaN leakage.


In [ ]:
with timed_section("Governance analysis"):

    # ── 1. Stale flag (vectorized, NaN-proof) ──────────────
    # Stale = not modified AND not used beyond the cutoff.
    # If we have usage data, require BOTH signals to be stale.
    # If we only have modification data, use that alone.
    dsm = df_final["days_since_modified"].fillna(99999)
    dsu = df_final["days_since_last_used"].fillna(99999)
    has_usage = df_final["last_used_date"].notna()

    mod_stale = dsm > stale_cutoff_days
    use_stale = dsu > stale_cutoff_days

    # np.select: first matching condition wins, default = False
    df_final["is_stale"] = np.select(
        [
            has_usage & mod_stale & use_stale,       # has usage, both stale → True
            has_usage & ~(mod_stale & use_stale),     # has usage, not both stale → False
            ~has_usage & mod_stale,                    # no usage, mod stale → True
            ~has_usage & ~mod_stale,                   # no usage, mod not stale → False
        ],
        [True, False, True, False],
        default=False
    )

    stale_count = df_final["is_stale"].sum()
    log.info(f"Stale items (>{stale_cutoff_days} days inactive): {stale_count}")

    # ── 2. Missing owner ───────────────────────────────────
    cb = df_final["created_by"].fillna("").astype(str).str.strip().str.lower()
    df_final["has_missing_owner"] = (cb == "") | (cb == "nan") | (cb == "none")
    log.info(f"Items with missing owner: {df_final['has_missing_owner'].sum()}")

    # ── 3. Duplicate names (null names excluded) ───────────
    df_final["is_duplicate_name"] = False
    has_name = df_final["name"].notna() & (df_final["name"].astype(str).str.strip() != "")
    if has_name.any():
        named = df_final.loc[has_name].copy()
        named["_key"] = (
            named["name"].str.lower().str.strip()
            + "||"
            + named["type"].str.lower().str.strip()
        )
        dup_keys = set(named["_key"].value_counts().pipe(lambda s: s[s > 1]).index)
        df_final.loc[has_name, "is_duplicate_name"] = named["_key"].isin(dup_keys)
    log.info(f"Items with duplicate name+type: {df_final['is_duplicate_name'].sum()}")

    # ── 4. Orphaned semantic models ────────────────────────
    report_names = set(
        df_final.loc[df_final["type"] == "Report", "name"]
        .dropna().str.lower().str.strip()
    )
    model_mask = df_final["type"] == "SemanticModel"
    df_final["is_orphaned_model"] = False
    if model_mask.any():
        df_final.loc[model_mask, "is_orphaned_model"] = ~df_final.loc[
            model_mask, "name"
        ].str.lower().str.strip().isin(report_names)
    log.info(f"Potentially orphaned semantic models: {df_final['is_orphaned_model'].sum()}")

    # ── 5. Orphaned SQL endpoints ──────────────────────────
    parent_names = set(
        df_final.loc[df_final["type"].isin(["Lakehouse", "Warehouse"]), "name"]
        .dropna().str.lower().str.strip()
    )
    endpoint_mask = df_final["type"] == "SQLEndpoint"
    df_final["is_orphaned_endpoint"] = False
    if endpoint_mask.any():
        df_final.loc[endpoint_mask, "is_orphaned_endpoint"] = ~df_final.loc[
            endpoint_mask, "name"
        ].str.lower().str.strip().isin(parent_names)
        orphaned_eps = df_final.loc[df_final["is_orphaned_endpoint"], "name"].tolist()
        if orphaned_eps:
            log.info(f"  Orphaned endpoint names: {orphaned_eps}")
    log.info(f"Potentially orphaned SQL endpoints: {df_final['is_orphaned_endpoint'].sum()}")

    # ── 6. Cleanup candidate score (0-100) ─────────────────
    df_final["cleanup_candidate_score"] = 0
    df_final.loc[df_final["is_stale"] == True, "cleanup_candidate_score"] += 30
    df_final.loc[df_final["is_unused_artifact"] == True, "cleanup_candidate_score"] += 25
    df_final.loc[df_final["has_missing_owner"] == True, "cleanup_candidate_score"] += 15
    df_final.loc[df_final["is_duplicate_name"] == True, "cleanup_candidate_score"] += 10
    df_final.loc[
        (df_final["is_orphaned_model"] == True) | (df_final["is_orphaned_endpoint"] == True),
        "cleanup_candidate_score"
    ] += 10
    df_final.loc[dsm > 180, "cleanup_candidate_score"] += 10

    high_score = (df_final["cleanup_candidate_score"] >= 50).sum()
    med_score  = ((df_final["cleanup_candidate_score"] >= 30)
                  & (df_final["cleanup_candidate_score"] < 50)).sum()
    log.info(f"Cleanup scores: {high_score} high-risk (>=50), {med_score} medium-risk (30-49)")


## 15. Stale items report

In [ ]:
with timed_section("Stale items report"):
    stale_df = df_final[df_final["is_stale"] == True].sort_values(
        "cleanup_candidate_score", ascending=False
    )
    if stale_df.empty:
        print(f"No stale items found (threshold: {stale_cutoff_days} days).")
    else:
        cols = ["name", "type", "created_by", "days_since_modified",
                "days_since_last_used", "cleanup_candidate_score"]
        cols = [c for c in cols if c in stale_df.columns]
        print(f"\nStale items (>{stale_cutoff_days} days inactive): {len(stale_df)}\n")
        display(stale_df[cols])


## 16. Top cleanup candidates

In [ ]:
with timed_section("Cleanup candidates"):
    top_candidates = df_final[
        df_final["cleanup_candidate_score"] >= 30
    ].sort_values("cleanup_candidate_score", ascending=False)

    if top_candidates.empty:
        print("No items scored >=30.")
    else:
        cols = ["name", "type", "created_by", "is_stale",
                "is_unused_artifact", "has_missing_owner",
                "is_duplicate_name", "is_orphaned_model",
                "is_orphaned_endpoint", "cleanup_candidate_score"]
        cols = [c for c in cols if c in top_candidates.columns]
        print(f"\nItems scoring >=30 (cleanup candidates): {len(top_candidates)}\n")
        display(top_candidates[cols].head(50))


## 17. Full inventory

In [ ]:
print(f"\nFull workspace inventory: {len(df_final)} items\n")
display(df_final)


## 18. Persist to Delta table

In [ ]:
with timed_section("Delta table persistence"):
    if not save_to_lakehouse:
        log.info("save_to_lakehouse=False — skipping.")
    elif df_final.empty:
        log.warning("No data to save.")
    else:
        try:
            output_columns = [
                "id", "name", "type", "description", "web_url",
                "workspace_id", "workspace_name", "capacity_id",
                "created_by", "modified_by",
                "created_date", "last_modified",
                "last_used_date", "access_count_30d", "unique_users_30d",
                "is_unused_artifact",
                "days_since_modified", "days_since_last_used",
                "state",
                "is_stale", "has_missing_owner", "is_duplicate_name",
                "is_orphaned_model", "is_orphaned_endpoint",
                "cleanup_candidate_score",
                "snapshot_id", "snapshot_time_utc",
            ]
            output_columns = [c for c in output_columns if c in df_final.columns]
            df_out = df_final[output_columns].copy()

            spark_df = spark.createDataFrame(df_out.astype(str))
            spark_df.write.mode("append").format("delta").saveAsTable(lakehouse_table_name)

            log.info(f"Snapshot appended to: {lakehouse_table_name}")
            log.info(f"  Snapshot ID: {snapshot_id}")
            log.info(f"  Rows written: {len(df_out)}")

        except Exception as e:
            log.error(f"Could not save to lakehouse: {e}")
            log.error("  Ensure a default Lakehouse is attached to this notebook.")


## 19. Execution summary

In [ ]:
print("=" * 70)
print("  FABRIC WORKSPACE INVENTORY v3.1 — EXECUTION SUMMARY")
print("=" * 70)
print(f"  Workspace:        {workspace_name} ({workspace_id})")
print(f"  Snapshot ID:      {snapshot_id}")
print(f"  Snapshot time:    {snapshot_time_utc}")
print(f"  Admin access:     {'Yes' if admin_available else 'No (limited metadata)'}")
print(f"  Scanner API:      created_date={df_final['created_date'].notna().sum()}, modified_by={df_final['modified_by'].notna().sum()}")
print(f"  Activity events:  {len(activity_data)} items with usage data")
print(f"  Unused artifacts: {len(unused_artifact_ids)} flagged, dates enriched for {len(unused_created_dates)} items")
print(f"  Activity API URL: {POWERBI_API_BASE}/admin/activityevents")
print(f"  ")
print(f"  INVENTORY TOTALS")
print(f"  ────────────────")
print(f"  Total items:      {len(df_final)}")
print(f"  Item types:       {df_final['type'].nunique()}")
print(f"  ")
print(f"  METADATA COVERAGE")
print(f"  ─────────────────")
for col in ["id", "name", "created_by", "modified_by", "created_date",
            "last_modified", "last_used_date", "web_url"]:
    if col in df_final.columns:
        n = df_final[col].notna().sum()
        pct = (n / len(df_final)) * 100
        print(f"  {col:25s} {n:4d}/{len(df_final)} ({pct:5.1f}%)")
print(f"  ")
print(f"  GOVERNANCE FLAGS")
print(f"  ────────────────")
print(f"  Stale items:              {(df_final['is_stale'] == True).sum()}")
print(f"  Not stale:                {(df_final['is_stale'] == False).sum()}")
print(f"  Missing owner:            {df_final['has_missing_owner'].sum()}")
print(f"  Duplicate names:          {df_final['is_duplicate_name'].sum()}")
print(f"  Orphaned models:          {df_final['is_orphaned_model'].sum()}")
print(f"  Orphaned SQL endpoints:   {df_final['is_orphaned_endpoint'].sum()}")
print(f"  Cleanup score >=50:       {(df_final['cleanup_candidate_score'] >= 50).sum()}")
print(f"  Cleanup score 30-49:      {((df_final['cleanup_candidate_score'] >= 30) & (df_final['cleanup_candidate_score'] < 50)).sum()}")
print(f"  ")
print(f"  SECTION TIMING")
print(f"  ──────────────")
total_time = 0
for section, elapsed in section_times.items():
    total_time += elapsed
    print(f"  {section:45s} {elapsed:6.1f}s")
print(f"  {'─' * 52}")
print(f"  {'Total':45s} {total_time:6.1f}s")
print(f"  ")
if save_to_lakehouse:
    print(f"  Delta table:      {lakehouse_table_name}")
print("=" * 70)


## 20. Schema reference

| Column | Source | Description |
|--------|--------|-------------|
| `id` | Core/Admin API | Fabric item unique identifier |
| `name` | Core/Admin API | Display name |
| `type` | Core/Admin API | Canonical Fabric item type |
| `description` | Core/Admin API | User-provided description |
| `web_url` | Constructed | Direct link to item in Fabric portal |
| `workspace_id` | Parameter | Workspace GUID |
| `workspace_name` | Workspace API | Workspace display name |
| `capacity_id` | Admin API | Fabric capacity GUID |
| `created_by` | Admin API | Creator UPN (via creatorPrincipal) |
| `modified_by` | Scanner API | Last modifier |
| `created_date` | Scanner API + Unused Artifacts | Creation timestamp |
| `last_modified` | Admin API | Last modification timestamp |
| `last_used_date` | Activity Events + Unused Artifacts | Last genuine access timestamp |
| `access_count_30d` | Activity Events | Event count in last 30 days |
| `unique_users_30d` | Activity Events | Distinct users in last 30 days |
| `is_unused_artifact` | semantic-link-labs | Flagged as unused by Power BI metrics |
| `days_since_modified` | Computed | Days since last_modified |
| `days_since_last_used` | Computed | Days since last_used_date |
| `state` | Admin API | Item state |
| `is_stale` | Governance | Inactive beyond stale_cutoff_days (True/False, never NaN) |
| `has_missing_owner` | Governance | created_by is null/empty |
| `is_duplicate_name` | Governance | Same name+type more than once (null names excluded) |
| `is_orphaned_model` | Governance | SemanticModel with no matching Report |
| `is_orphaned_endpoint` | Governance | SQLEndpoint with no matching Lakehouse/Warehouse |
| `cleanup_candidate_score` | Governance | 0-100 composite risk score |

### Important notes

1. **`last_modified` ≠ `last_used`** — Completely separate signals.
2. **`created_date` and `modified_by`** require Scanner API (`semantic-link-labs`).
3. **Unused artifacts** also provide `created_date` and `last_accessed` for ~40 Power BI items (epoch ms converted to ISO).
4. **Activity Events** cover max 30 days and primarily track Power BI item interactions (report views, dataset refreshes).
5. **This notebook is read-only** — no Fabric items are created, modified, or deleted.
